# Earth Departure

Goals:
- Departure Geometry (rla, dla)
- Escape orbit and burn
- Parking orbit and location
- Launch site and opportunities
- Propagation

In [106]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [107]:
import numpy as np
from cesium.constants import (
    HOUR_TO_SEC,
    KM_TO_M,
    M_TO_KM,
    RAD_TO_DEG,
)
from cesium.guidance.interplanetary import (
    EarthDepartureTrajectory,
    interplanetary_circ_injection,
)
from cesium.math.base import unit
from cesium.math.time import jd_to_greg
from cesium.orbital.interplanetary import hyperbolic_characteristics, sph_of_infl
from cesium.orbital.transformations import (
    calc_dcm_eci_wrt_hci,
    geodetic_LLA_to_ECEF,
)
from cesium.plotting import plot_3d
from cesium.world.constants import MU_EARTH, MU_SUN, R_EARTH
from numpy.lib.npyio import NpzFile
from numpy.linalg import norm

## Loading trajectory
From `interplanetary.ipynb`

In [108]:
MISSION_NAME = "birthday-christmas-earlier"
mission: NpzFile = np.load(f"trajectories/{MISSION_NAME}.mission", allow_pickle=False)


def print_mission(mission: NpzFile):
    """Print dates, duration, sampling, units, and stored fields from a mission archive.

    Parameters
    ----------
    mission : NpzFile
        Loaded Mars Courier .mission archive in NumPy NPZ format. Expected
        fields include lambert_departure_jd, lambert_arrival_jd, tof_s,
        time_s, dt_s, frame, position_units, velocity_units, and time_origin.
        Epochs are heliocentric Lambert references, not liftoff or SOI crossings.

    Notes
    -----
    Uses the notebook's MISSION_NAME as the display title. State-array shapes
    and dtypes are listed without changing or closing the loaded archive.
    """
    print(f"Mission: {MISSION_NAME}")
    print("Heliocentric reference times")
    for label, key in (
        ("Lambert departure (Earth reference)", "lambert_departure_jd"),
        ("Lambert arrival (Mars reference)", "lambert_arrival_jd"),
    ):
        print(
            f"{label}: "
            + "{:04.0f}-{:02.0f}-{:02.0f} {:02.0f}:{:02.0f}:{:06.3f}".format(
                *jd_to_greg(float(mission[key]))
            )
        )
    print(f"Flight duration: {float(mission['tof_s']) / 86400:.4f} days")
    print(
        f"Samples: {mission['time_s'].size} (seconds since {mission['time_origin'].item()})"
    )
    # print(f"Timesteps ranging from {mission['dt_s'].min():.3f}-{mission['dt_s'].max():.3f} s ")
    print(
        f"Frame: {mission['frame'].item()}; positions: {mission['position_units'].item()}; velocities: {mission['velocity_units'].item()}"
    )
    print("\nStored fields:")
    for key in mission.files:
        info = mission[key].dtype

        if mission[key].dtype != float:
            info = '"' + mission[key] + '"'
        elif mission[key].shape == ():
            info = mission[key]

        # info = mission[key].dtype if mission[key].dtype == float else mission[key]
        print(f"  {key:24s} {mission[key].shape!s:14s} {info}")


print_mission(mission)


rv_sat_eci_km = (mission["satellite_rv_hci_km"] - mission["earth_rv_hci_km"]) @ np.kron(
    np.eye(2), calc_dcm_eci_wrt_hci()
).T
r_lambert_start_eci_km = rv_sat_eci_km[0, 0:3]
v_inf_eci_km = rv_sat_eci_km[0, 3:6]
lambert_dep_jd = float(mission["lambert_departure_jd"])

Mission: birthday-christmas-earlier
Heliocentric reference times
Lambert departure (Earth reference): 2035-06-14 12:00:00.000
Lambert arrival (Mars reference): 2035-12-25 00:00:00.000
Flight duration: 193.5000 days
Samples: 1936 (seconds since lambert_departure)
Frame: HCI; positions: km; velocities: km/s

Stored fields:
  lambert_departure_jd     ()             2464493.0
  lambert_arrival_jd       ()             2464686.5
  tof_s                    ()             16718400.0
  time_s                   (1936,)        float64
  dt_s                     (1935,)        float64
  earth_rv_hci_km          (1936, 6)      float64
  mars_rv_hci_km           (1936, 6)      float64
  satellite_rv_hci_km      (1936, 6)      float64
  frame                    ()             "HCI"
  position_units           ()             "km"
  velocity_units           ()             "km/s"
  time_origin              ()             "lambert_departure"
  trajectory_model         ()             "heliocentric_two_body

In [109]:
# ECEF stuff. Launch site position at Lambert departure


# Cape Canaveral
LAUNCH_LAT_DEG = 28.48889
LAUNCH_LON_DEG = -80.57778
r_cape_ecef_m = geodetic_LLA_to_ECEF(LAUNCH_LAT_DEG, LAUNCH_LON_DEG)

# Orbit
PARKING_ORBIT_ALT_M = 300e3  # m
earth_soi_km = sph_of_infl(norm(mission["earth_rv_hci_km"][0, 0:3]), MU_EARTH, MU_SUN)

The plot below has the Sun vector for reference. Here's where you can really see that it's 11:25p in June.

## Departure Characteristics

In [110]:
# ECI
C3_km2, rla, dla = hyperbolic_characteristics(v_inf_eci_km)


PERIAPSE = R_EARTH + PARKING_ORBIT_ALT_M
dv_p, ta_asymptote, aiming_radius, e = interplanetary_circ_injection(
    norm(v_inf_eci_km * KM_TO_M), PERIAPSE, MU_EARTH, deg=False
)

print("Departure target (Lambert reference epoch)")
print(f"  v_inf: {norm(v_inf_eci_km):.4f} km/s; C3: {C3_km2:.4f} km²/s²")
print(f"  RLA: {rla * RAD_TO_DEG:.4f} deg; DLA: {dla * RAD_TO_DEG:.4f} deg")
print("\nCircular parking orbit / ideal tangential escape burn")
print(f"  Parking altitude: {PARKING_ORBIT_ALT_M * M_TO_KM:.1f} km")
print(f"  Periapsis radius: {PERIAPSE * M_TO_KM:.3f} km from Earth's center")
print(f"  Circular speed: {np.sqrt(MU_EARTH / PERIAPSE) * M_TO_KM:.4f} km/s")
print(
    f"  Post-burn periapsis speed: {np.sqrt(norm(v_inf_eci_km * KM_TO_M) ** 2 + 2 * MU_EARTH / PERIAPSE) * M_TO_KM:.4f} km/s"
)
print(f"  Required burn delta-v: {dv_p * M_TO_KM:.4f} km/s")
print("\nEscape hyperbola")
print(f"  Eccentricity: {e:.6f}")
print(f"  Asymptotic true anomaly: {ta_asymptote * RAD_TO_DEG:.4f} deg")
print(f"  Aiming radius (impact parameter): {aiming_radius * M_TO_KM:.3f} km")

Departure target (Lambert reference epoch)
  v_inf: 3.2581 km/s; C3: 10.6155 km²/s²
  RLA: 4.7883 deg; DLA: 4.0568 deg

Circular parking orbit / ideal tangential escape burn
  Parking altitude: 300.0 km
  Periapsis radius: 6678.138 km from Earth's center
  Circular speed: 7.7258 km/s
  Post-burn periapsis speed: 11.4013 km/s
  Required burn delta-v: 3.6756 km/s

Escape hyperbola
  Eccentricity: 1.177852
  Asymptotic true anomaly: 148.1034 deg
  Aiming radius (impact parameter): 23368.986 km


## Targeting escape
Want to stitch trajectories by time

In [111]:
departure = EarthDepartureTrajectory(
    lambert_dep_jd, v_inf_eci_km, PERIAPSE, r_cape_ecef_m
)
departure.launch_that_babyy(tspan_s=12 * HOUR_TO_SEC, npoints=100000)
departure.evaluate()
departure.midcourse_correction(
    lambert_dep_jd, mission["time_s"], rv_sat_eci_km, mission["earth_rv_hci_km"]
)
print()

Launching on (2035, 6, 14, 12, 0, 0.0)
evaluate()
  Perpendicular distance to v_inf comparison (km)
  h/v_inf: 23368.985861 (0.000019 km s.d. very small)
  Theoretical aiming radius [km]: 23368.985853
  diff [m]: 0.00793830406870693 (0.01861948049095884 s.d. very small)
midcourse_correction()
  losest Lambert sample: 7; date: (2035, 6, 15, 4, 48, 1.6093253929483353e-05)
  Earth-relative position separation: 21638.366 km
  Lambert sample time seconds behind ECI propagated time: 17280.0 s
  Velocity-only correction HCI [km/s]: [-0.56202614  0.04752943  0.01205081]; magnitude: 564.161 m/s
  This compares the nearest stored sample; a continuous handoff also requires matching position and epoch.
Crossed sphere of influence: False



### Offset to match correction burn

In [112]:
actual_launch_jd = lambert_dep_jd + 0.42
departure = EarthDepartureTrajectory(
    actual_launch_jd, v_inf_eci_km, PERIAPSE, r_cape_ecef_m
)
departure.launch_that_babyy(tspan_s=71.5 * HOUR_TO_SEC, npoints=100000)
departure.evaluate()
departure.midcourse_correction(
    lambert_dep_jd, mission["time_s"], rv_sat_eci_km, mission["earth_rv_hci_km"]
)
print()

Launching on (2035, 6, 14, 22, 4, 47.999993562698364)
evaluate()
  Perpendicular distance to v_inf comparison (km)
  h/v_inf: 23368.985863 (0.000068 km s.d. very small)
  Theoretical aiming radius [km]: 23368.985853
  diff [m]: 0.01089497874379158 (0.06775641162343336 s.d. very small)
midcourse_correction()
  losest Lambert sample: 34; date: (2035, 6, 17, 21, 35, 59.99999195337282)
  Earth-relative position separation: 26285.913 km
  Lambert sample time seconds behind ECI propagated time: 72.0 s
  Velocity-only correction HCI [km/s]: [-0.13145392 -0.00904612 -0.00354297]; magnitude: 131.812 m/s
  This compares the nearest stored sample; a continuous handoff also requires matching position and epoch.
Crossed sphere of influence: True



## Visualization

In [113]:
# Translate by Earth's rv at each sample, then rotate HCI into ECI.
sun_direction_eci = calc_dcm_eci_wrt_hci() @ (-mission["earth_rv_hci_km"][0, :3])


frames = [
    {
        "name": "ECI",
        "location": [0, 0, 0],
        "dcm_body_wrt_hci": np.eye(3),
        "length": 10000,
    },
    # {"name": "ECEF", "location": [0, 0, 0],
    #         "dcm_body_wrt_hci": dcm_ecef_wrt_eci, "length": 10000},  # just for visualization
]

vectors = [
    {
        "direction": sun_direction_eci,
        "length": 20000,
        "name": "Sun direction",
        "color": "goldenrod",
        "label": True,
    },
    {
        "direction": departure.r_p_eci,
        "length": 10000,
        "name": "periapse",
        "color": "orange",
        "label": True,
    },
    {
        "direction": departure.n_vec,
        "length": 10000,
        "name": "Normal vec",
        "color": "black",
        "label": True,
    },  # just for visualization
    {
        "direction": departure.right_vec,
        "length": 10000,
        "name": "right_vec",
        "color": "black",
        "label": True,
    },  # just for visualization
]

points = [
    {
        "name": "Cape Canaveral",
        "location": (departure.dcm_ecef_wrt_eci0.T @ r_cape_ecef_m) * M_TO_KM,
        "color": "red",
    },
    {"name": "Periapsis", "location": departure.r_p_eci * M_TO_KM, "color": "orange"},
    {
        "name": "SOI along v_inf",
        "location": earth_soi_km * unit(v_inf_eci_km),
        "color": "green",
    },
]

view_range = 1000000
fig_earth = plot_3d(
    [rv_sat_eci_km[:, 0:3], departure.rv_traj_eci_m[:, 0:3] * M_TO_KM],
    [mission["time_s"], departure.t_traj_s],
    labels=["Lambert Departure Direction", "Trajectory"],
    trace_colors=["purple", "black"],
    bodies={"name": "Earth", "radius": R_EARTH * M_TO_KM, "color": "royalblue"},
    frames=frames,
    vectors=vectors,
    points=points,
    title=f"Earth departure — {view_range} km view",
    units="km",
    azim=80,
    elev=30,
    show_end=False,
    show=False,
)
fig_earth.update_scenes(
    xaxis_range=[-view_range, view_range],  # km
    yaxis_range=[-view_range, view_range],  # km
    zaxis_range=[-view_range, view_range],  # km
)
fig_earth.show()

## Validating

In [114]:
# Impact parameter / aiming distance

# h/v_inf
impact_parameter = norm(
    np.cross(departure.rv_traj_eci_m[:, :3], departure.rv_traj_eci_m[:, 3:]), axis=1
) / norm(v_inf_eci_km * KM_TO_M)
print("Perpendicular distance to v_inf comparison (km)")
print(
    f"  h/v_inf: {impact_parameter.mean() * M_TO_KM:.6f} ({impact_parameter.std() * M_TO_KM:.6f} km s.d. very small)"
)
print(f"  Theoretical aiming radius [km]: {aiming_radius * M_TO_KM:.6f}")
diff = impact_parameter - aiming_radius
print(f"  diff [m]: {diff.mean()} ({diff.std()} s.d. very small)")

Perpendicular distance to v_inf comparison (km)
  h/v_inf: 23368.985863 (0.000068 km s.d. very small)
  Theoretical aiming radius [km]: 23368.985853
  diff [m]: 0.01089497874379158 (0.06775641162343336 s.d. very small)


In [115]:
# C3 at each point
c3_propagated = (
    np.array(
        norm(departure.rv_traj_eci_m[:, 3:6], axis=1) ** 2
        - 2 * MU_EARTH / norm(departure.rv_traj_eci_m[:, 0:3], axis=1)
    )
    * M_TO_KM**2
)

diff_c3 = c3_propagated - C3_km2

print(
    f"Diff between `c3_propagated` and `c3_lambert` [km2]: {diff_c3.mean():.6e} ({diff_c3.std():.6e} s.d.)"
)

Diff between `c3_propagated` and `c3_lambert` [km2]: 3.598526e-08 (3.090403e-08 s.d.)


## Burns
Propagated to 12 hours because it would take forever to get to the sphere of influence

### Circular

## Save departure trajectory

Export Earth-centered ECI position/velocity in meters and m/s, elapsed seconds,
and the absolute JD at the modeled periapsis burn. Run `ground_segment.ipynb`
to analyze station visibility and communications without repeating propagation.


In [116]:
from pathlib import Path

# t=0 is this modeled periapsis burn, not surface liftoff.
departure_path = Path(f"trajectories/{MISSION_NAME}-departure.mission")
departure_path.parent.mkdir(parents=True, exist_ok=True)
assert departure.rv_traj_eci_m.shape == (len(departure.t_traj_s), 6)
assert np.isfinite(departure.rv_traj_eci_m).all()
with departure_path.open("wb") as file:
    np.savez_compressed(
        file,
        epoch_jd=departure.launch_date_jd,
        time_s=departure.t_traj_s,
        rv_eci_m=departure.rv_traj_eci_m,
        launch_site_lla_deg_m=np.array([LAUNCH_LAT_DEG, LAUNCH_LON_DEG, 0.0]),
        v_inf_eci_km_s=departure.v_inf_eci_km,
        source_mission=MISSION_NAME,
        frame="ECI",
        position_units="m",
        velocity_units="m/s",
        time_origin="periapsis_burn",
        time_units="s",
    )
print(departure_path)

trajectories/birthday-christmas-earlier-departure.mission
